# 2 — The funding carry (cash-and-carry) — flagship strategy

**Thesis.** Perpetual funding is not merely a *cost* on a leveraged long; when
it is persistently positive it is a **harvestable carry**. Holding **+1 BTC spot
and −1 BTC perp** (notional-matched) nets price exposure to ~zero (only the tiny,
mean-reverting basis remains) while the short perp leg **receives** funding.

The inputs this trade needs — `fundingRate`, `markPrice`, spot and perp closes —
are already in the snapshot. We build the carry book and measure it like a
strategy, rather than treating funding only as a drag on a directional long.

In [1]:
import warnings

import pandas as pd
import plotly.io as pio

from perp_spot import backtest, config, data, funding, plots, strategy

pio.renderers.default = "notebook_connected"  # embed plotly.js for static HTML export
warnings.simplefilter("ignore")
market = data.load_market(config.PRIMARY_SYMBOL)
daily, fund = market.daily, market.funding

## 2.1 Funding is a sizeable, time-varying carry

Quoted correctly: as an **APR on notional** and in **bps/8h** (both
leverage-invariant). Funding accrues on the notional (~$4k at 4×), not the
posted margin, so a margin-relative figure overstates it ~4×.

In [2]:
print(f"Mean funding: {fund['funding_rate'].mean() * 100:.4f}% / 8h "
      f"→ {funding.funding_apr(fund) * 100:.1f}% APR (full history)")
print(f"Share of 8h settlements with POSITIVE funding (longs pay): "
      f"{(fund['funding_rate'] > 0).mean():.1%}")

wf = backtest.walk_forward(market, freq="QE")
apr_by_q = wf.groupby("window_start")["funding_apr"].first()
print(f"Quarterly funding APR ranges {apr_by_q.min() * 100:.0f}%..{apr_by_q.max() * 100:.0f}% "
      f"(median {apr_by_q.median() * 100:.0f}%) — strongly regime-dependent.")

# Basis (top) and rolling annualized funding (bottom) as a positioning signal:
plots.fig_basis(daily).show()

Mean funding: 0.0125% / 8h → 13.6% APR (full history)
Share of 8h settlements with POSITIVE funding (longs pay): 88.0%
Quarterly funding APR ranges 3%..66% (median 8%) — strongly regime-dependent.


## 2.2 The carry book over the full history (2021–2024)

One delta-neutral position held across all regimes. The equity curve is the
accumulated funding income (plus basis convergence, minus round-trip fees).

In [3]:
window = funding.holding_window(daily, config.HISTORY_START, config.HISTORY_END)
carry = strategy.carry_delta_neutral(daily, fund, window, capital=config.DEFAULT_INVESTMENT)
spot_full = strategy.long_spot(daily, window, config.DEFAULT_INVESTMENT)

from perp_spot import metrics
print("Carry (Δ-neutral) over 2021–2024:")
print(metrics.summarize(carry.equity, name="Carry").to_string())
print(f"\nTotal funding income: ${carry.funding_total:,.2f} on ${config.DEFAULT_INVESTMENT:,.0f} capital")
print(f"Carry max drawdown {metrics.max_drawdown(carry.equity):.2%} vs "
      f"buy-and-hold spot {metrics.max_drawdown(spot_full.equity):.2%}")

plots.fig_equity_curves({"spot": spot_full, "carry": carry}).show()
plots.fig_cumulative_funding(
    funding.accrue_funding(fund, carry.meta["size"], window).series
).show()

Carry (Δ-neutral) over 2021–2024:
total_return     0.941779
ann_return       0.180322
ann_vol          0.018780
sharpe           8.839344
sortino         32.540792
max_drawdown    -0.003461
calmar          52.108204
var_95          -0.000229
cvar_95         -0.000444
hit_rate         0.789870

Total funding income: $942.99 on $1,000 capital
Carry max drawdown -0.35% vs buy-and-hold spot -76.69%


## 2.3 Carry vs the directional books, side by side

Same window, three books: 1× spot, 4× directional perp (net of funding), and the
delta-neutral carry.

In [4]:
books = backtest.run_books(market, config.CASE_START, config.CASE_END)
table = backtest.metrics_table(books)[
    ["total_return", "ann_vol", "sharpe", "sortino", "max_drawdown", "funding_total", "liquidated"]
]
table.round(3)

,total_return,ann_vol,sharpe,sortino,max_drawdown,funding_total,liquidated
Spot 1x,0.602,0.620,3.784,4.040,-0.158,0.000,False
Perp 4x,2.152,1.773,3.850,4.115,-0.505,260.688,False
Carry (Δ-neutral),0.062,0.018,15.154,10.958,-0.003,65.219,False


## 2.4 Is the carry robust across regimes?

The single-window number could be luck. We run every non-overlapping quarter in
2021–2024 and report the **distribution** of carry returns vs the directional
books — with a block-bootstrap confidence interval on the carry's mean.

In [5]:
carry_q = wf[wf["strategy"] == "Carry (Δ-neutral)"]["total_return"]
print(f"Carry: profitable in {(carry_q > 0).sum()}/{len(carry_q)} quarters; "
      f"median {carry_q.median():.2%}, worst {carry_q.min():.2%}")
bs = backtest.block_bootstrap(carry_q, statistic=lambda s: s.mean(), block_size=2, n_resamples=2000)
print(f"Bootstrap mean quarterly carry return: {bs['point']:.2%} "
      f"[95% CI {bs['lo']:.2%}, {bs['hi']:.2%}]")

plots.fig_walkforward(wf, "total_return").show()

Carry: profitable in 16/16 quarters; median 2.23%, worst 0.45%
Bootstrap mean quarterly carry return: 4.12% [95% CI 1.58%, 6.38%]


### Takeaways
1. BTC funding has been a **double-digit APR** carry on average, but it is
   regime-dependent (it compresses, and occasionally inverts, in bear markets).
2. The delta-neutral carry harvested it with a **near-zero drawdown** and was
   profitable in essentially every quarter — a genuine risk-managed carry, not a
   tax on a leveraged bet.
3. The framing that matters is asking "is funding a harvestable carry?" rather
   than only "what does funding cost a directional long?".